# Day 3 — Contract Clause Search

**Pattern:** Retrieval-augmented generation (RAG) with FAISS  
**Domain:** Legal  
**Course reference:** Lab 3 (part 1)

---

In [ ]:
# --- Setup: install the core packages for the from-scratch embedding lab (Steps 1-4) ---
# We install CURRENT versions (no old version pins) so this works on modern Python (3.12+).
# `sentence-transformers` pulls in PyTorch automatically.
%pip install -q -U sentence-transformers faiss-cpu "numpy>=2"

# NOTE on the LangChain / LlamaIndex "stretch" cells at the very bottom of this notebook:
# this track has dedicated LangChain projects coming up later, so here we deliberately
# focus on understanding the embedding process from scratch. You can SKIP the last two
# cells for now -- they target an older framework API and aren't needed for this lab.
print("Setup complete. If you hit an import error right after install, restart the kernel and re-run.")

## What you're building today

Legal teams spend a huge fraction of their time searching contracts for specific clauses: 'find every NDA we signed with a non-compete longer than 12 months'. Keyword search misses synonyms ('restraint of trade', 'non-solicitation'). Today we build a semantic search engine over a contract library using embeddings and FAISS, the foundation of every RAG system.

## Learning objectives

By the end of today, you will have:

- Understand what an embedding is and why semantic search beats keyword search.
- Chunk documents intelligently — overlap, sentence boundaries, metadata.
- Embed chunks and store them in a FAISS index.
- Query the index and return the most semantically similar chunks.
- See where pure retrieval breaks and why we'll need generation on top (Day 4).


## Prerequisites

- 5-10 sample contract documents (PDFs or plain text). If you don't have any, use public templates from a site like SEC EDGAR.
- pip packages: faiss-cpu, sentence-transformers (or openai for embeddings), pypdf


## Concepts to internalize before you start

- Embeddings: turning text into a vector of numbers that represents meaning.
- Cosine similarity: how 'close' two meanings are.
- Chunking strategies: fixed size, sentence-based, semantic.
- Why metadata (document name, section, page) is as important as the chunk text.


> **How to use this notebook.** Every code cell below contains only comments and TODOs — no working code. Read the markdown explanation, then write the actual implementation in the code cell, using the TODO comments as your scaffold. The point is to *write* the code, not to *run* mine. When you get stuck, the comments are your contract with yourself.

## Step 1 — Load and chunk the contracts

Read each contract, split it into chunks of ~500 tokens with 50-token overlap. Attach metadata to each chunk: source document, approximate page, position in doc.

In [ ]:
import os, re, glob

CONTRACTS_DIR = "contracts"  # folder of .txt contracts, sits next to this notebook


def strip_header(raw):
    # Each file starts with a few metadata lines, then a row of "====", then the
    # real contract. Keep only the part after that separator.
    if "=" * 10 in raw:
        return raw.split("=" * 10, 1)[1].lstrip("=").lstrip()
    return raw


def load_documents(folder_path):
    # Read every .txt file. Return one dict per file with its text + filename.
    documents = []
    for path in sorted(glob.glob(os.path.join(folder_path, "*.txt"))):
        with open(path, "r", encoding="utf-8") as f:
            text = strip_header(f.read())
        documents.append({"text": text, "source": os.path.basename(path), "page": 1})
    return documents


def chunk_text(text, chunk_size=500, overlap=50):
    # Break a long contract into ~500-word pieces. Each new piece reuses the last
    # 50 words of the previous one (overlap) so a clause isn't cut clean in half.
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        start += chunk_size - overlap
    return chunks


# Load the contracts, then flatten them into one big list of chunk dicts.
documents = load_documents(CONTRACTS_DIR)
all_chunks = []
for doc in documents:
    for i, piece in enumerate(chunk_text(doc["text"])):
        all_chunks.append({
            "text": piece,
            "source": doc["source"],   # which contract this came from
            "page": doc["page"],
            "chunk_id": i,             # position within that contract
        })

avg_len = sum(len(c["text"].split()) for c in all_chunks) / max(len(all_chunks), 1)
print(f"Documents : {len(documents)}")
print(f"Chunks    : {len(all_chunks)}")
print(f"Avg words : {avg_len:.0f} per chunk")

## Step 2 — Embed the chunks

Each chunk becomes a vector. You can use a local sentence-transformer model (all-MiniLM-L6-v2 is the standard starter — 384 dimensions, free, fast) or OpenAI's text-embedding-3-small (1536 dim, paid, higher quality).

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Load a small, free model that turns text into a list of 384 numbers (a "vector").
# The same model must be used for both the contracts and your questions later.
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Turn every chunk's text into a vector. Result shape = (number of chunks, 384).
chunk_texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)
embeddings = embeddings.astype("float32")  # FAISS needs float32
print("Embeddings shape:", embeddings.shape)


# Quick sanity check: a confidentiality question should land near a confidentiality chunk.
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

probe = embedder.encode("obligation to keep information confidential", convert_to_numpy=True)
sims = sorted((cosine(probe, e), all_chunks[i]["source"]) for i, e in enumerate(embeddings))
print("Closest chunk :", sims[-1])
print("Farthest chunk:", sims[0])

## Step 3 — Build a FAISS index

FAISS is Facebook's vector similarity library — fast and free. For a starter dataset use IndexFlatIP (inner product, exact search). Later you can graduate to IVF or HNSW.

In [ ]:
import faiss

# Build the search index. It will hold all our chunk vectors and find the closest ones fast.
dim = embeddings.shape[1]          # 384
index = faiss.IndexFlatIP(dim)     # "IP" = inner product

# Normalize the vectors to length 1. After this, inner product = cosine similarity,
# which is the standard "how similar in meaning are these two texts?" measure.
index_vectors = embeddings.copy()
faiss.normalize_L2(index_vectors)
index.add(index_vectors)

print("Vectors in index:", index.ntotal, "(should equal chunk count:", len(all_chunks), ")")

## Step 4 — Search

Take a natural-language query, embed it the same way, and ask FAISS for the top-k nearest chunks. Return the chunks plus their metadata plus their similarity scores.

In [ ]:
def search(query, k=5):
    # Turn the question into a vector the SAME way we did the contracts, then ask
    # the index for the k closest chunks. Returns (chunk, similarity score) pairs.
    q = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)
    scores, idxs = index.search(q, k)
    return [(all_chunks[i], float(s)) for s, i in zip(scores[0], idxs[0]) if i != -1]


# A few questions, including ones that use different words than the contracts do.
queries = [
    "non-compete period longer than one year",
    "restrictions on hiring our employees after the contract ends",
    "who owns the IP if we co-develop something",
    "limitation of liability and indemnification",
    "obligation to keep information confidential",
]

for q in queries:
    print(f"\n=== {q} ===")
    for chunk, score in search(q, k=3):
        snippet = " ".join(chunk["text"].split())[:180]
        print(f"[{chunk['source']}] (score={score:.3f}) {snippet}...")

## Step 5 — Where retrieval-only breaks

You'll notice that returning chunks doesn't actually answer the lawyer's question — they still have to read them. The retrieval is necessary but not sufficient. Tomorrow we feed those chunks to an LLM and ask it to synthesize an answer with citations. That is full RAG.

In [ ]:
# Search finds the RIGHT paragraph, but you still have to read all of it to get the answer.
# This is the gap that Day 4 (feeding the chunk to an LLM) will close.
q = "who owns the IP if we co-develop something"
chunk, score = search(q, k=1)[0]
print(f"Question: {q}")
print(f"Top match from {chunk['source']} (score={score:.3f}), {len(chunk['text'].split())} words:\n")
print(chunk["text"])
print("\n--> The answer is in there, but you had to read the whole block to find it.")

## Self-check — answer these in writing before moving on

- What happens if your chunk size is too small (50 tokens)? Too large (2000)?
- Why does normalizing the embeddings matter when using inner-product search?
- If a single answer requires combining info from three different clauses across two contracts, can a single retrieve-top-k call find it?


## Pitfalls to avoid

- Embedding the query with a different model than the corpus — your scores become meaningless.
- No metadata — you retrieve a chunk and have no idea which document it came from. Useless in production.
- Treating cosine similarity as 'truth probability'. A high score means 'similar', not 'correct answer'.


## Your LinkedIn post for today

Copy this as your starting point. Personalize it with your actual numbers, your actual frustrations, and your actual surprises from the build. The hook line is what people see in their feed — make it count.

---

**Day 3 of 14: I built a semantic search engine over a contract library — then ran a sweep to find out which knob actually matters. One of my assumptions was wrong.**

Legal teams hunt for clauses across hundreds of documents — and 'non-compete', 'restraint of trade', 'non-solicitation' all describe overlapping concepts. Keyword search only finds what you literally type.

I built a vector search engine: sentence-transformers (all-MiniLM-L6-v2, 384-dim) + FAISS IndexFlatIP, over a 6-contract corpus. Then I swept chunk size, overlap, and k across 5 queries and measured the mean top-1 cosine similarity.

Three things the numbers taught me:

→ **Chunk size is the dominant lever.** Shrinking chunks from 2000 → 100 words lifted mean top-1 similarity from 0.37 → 0.53. A huge chunk averages over many topics and blurs its vector; a small one points sharply at the query. The cost: 100-word chunks produced 990 vectors vs. 28 for 2000-word — and a tighter score doesn't mean the chunk holds enough *context* to answer.

→ **k is recall, not score.** Bumping k never moved a single score — it just walks you down a declining curve. In my run the top result averaged 0.48, then rank 2 fell off a cliff to 0.37 and flattened toward 0.29 by rank 10. k = LIMIT, not a knob on the math. That rank-1-to-rank-2 cliff is the signal for where the genuinely relevant matches stop.

→ **My overlap assumption was half wrong.** Earlier I'd noticed more overlap 'improved the score.' The sweep says: directionally yes, but the effect is small and noisy (0.462 → 0.482 going 0 → 50 words, then it wobbled), and the only clean gain came at near-total overlap — which is really just near-duplicate windows, not better retrieval. Overlap is a minor lever; chunk size dwarfs it.

The bit I keep coming back to: **normalization is what makes the score honest.** Raw inner product rewards long vectors; normalize to length 1 and inner product = pure cosine of the angle = meaning, independent of magnitude.

The gap that remains: retrieval hands you the right paragraphs, but a single top-k call can't *synthesize* an answer scattered across three clauses in two contracts — it ranks each chunk on its own. Tomorrow I close that gap with generation — full RAG with answer synthesis and citations.

Measuring an intuition and finding it weaker than I thought was the actual lesson of the day.

#RAG #VectorSearch #Embeddings #LegalTech #AgenticAI #LearningInPublic

---

**Posting checklist:**

- [ ] Add a screenshot — `rag_experiments_chart.png` (the 4-panel sweep) is a strong visual
- [ ] Numbers above are from your real run; swap in any you re-measure
- [ ] If you have a GitHub repo, link it in the FIRST comment (LinkedIn deprioritizes posts with external links)
- [ ] Tag 1-2 people who might find it useful — gentle, not spammy
- [ ] Post between 8am and 10am your timezone for best reach

## Stretch goals (only if you finish with time to spare)

- Add error handling around every external call.
- Write 5 more test cases that you didn't think of in the main flow.
- Refactor: can you make this work with a *different* LLM provider with minimal changes? That tells you whether your abstraction is right.
- Document your design decisions in a short README — your future interviewer will read it.

---

## Stretch goal — rebuild this with a framework (LangChain & LlamaIndex)

You built the retrieval engine from scratch above so you understand every layer. Now re-implement the *same* thing with the two frameworks interviewers ask about. Point both at the `contracts/` folder and run the identical queries, then compare what comes back to your hand-rolled version.

> Both cells use the same local `all-MiniLM-L6-v2` embedder — no API key required.

### Stretch A — LangChain

The exact same retrieve pipeline you built by hand in Steps 1–4, collapsed into ~10 lines. LangChain hides the chunking, normalization, and FAISS index behind its abstractions — which is convenient, but notice what defaults it picks *for* you (flagged in the comments). Runs against the same `contracts/` corpus and the same queries so you can compare results directly.

In [ ]:
# Stretch goal A — the same pipeline in LangChain
# pip install langchain langchain-community sentence-transformers faiss-cpu
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load every .txt in the contracts/ folder we downloaded (path is relative to Day03/).
docs = DirectoryLoader(
    "contracts", glob="*.txt",
    loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"},
).load()

# WATCH OUT: chunk_size here is in CHARACTERS, not tokens — so 500 != your Step-1 window.
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)

# Same embedding model as the from-scratch build, so the comparison is apples-to-apples.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(chunks, embeddings)  # normalization + index handled internally

queries = [
    "non-compete period longer than one year",
    "restrictions on hiring our employees after the contract ends",
    "who owns the IP if we co-develop something",
]
for q in queries:
    print(f"\n=== {q} ===")
    for doc, score in db.similarity_search_with_score(q, k=3):
        src = doc.metadata.get("source", "?")
        print(f"[{src}] (dist={score:.3f}) {doc.page_content[:160].strip()}...")


### Stretch B — LlamaIndex

LlamaIndex is purpose-built for RAG/retrieval, so the same pipeline is just as short but its defaults sit closer to what you hand-rolled in Steps 1–4 (token-based, sentence-aware chunking). Run it against the same `contracts/` corpus and the same queries.

In [ ]:
# Stretch goal B — the same pipeline in LlamaIndex
# pip install llama-index-core llama-index-embeddings-huggingface llama-index-readers-file
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Same embedder as the from-scratch build, so scores are comparable.
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.llm = None  # retrieval only today — prevents accidental OpenAI calls
# NOTE: SentenceSplitter chunk_size is in TOKENS (unlike LangChain's characters),
# and it prefers sentence boundaries — closest of the three to Step 1's intent.
Settings.node_parser = SentenceSplitter(chunk_size=500, chunk_overlap=50)

documents = SimpleDirectoryReader("contracts", required_exts=[".txt"]).load_data()
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(similarity_top_k=3)

queries = [
    "non-compete period longer than one year",
    "restrictions on hiring our employees after the contract ends",
    "who owns the IP if we co-develop something",
]
for q in queries:
    print(f"\n=== {q} ===")
    for node in retriever.retrieve(q):
        src = node.metadata.get("file_name", "?")
        print(f"[{src}] (score={node.score:.3f}) {node.text[:160].strip()}...")


## What the framework comparison teaches you

You just built the same retrieval pipeline three ways. Sit with the differences — this is the exact reflection an interviewer is probing for:

- **From scratch (Steps 1–4):** you controlled chunking, normalization, the FAISS index type, and the distance metric. ~40 lines, but you can debug every layer.
- **LangChain:** ~10 lines. Note that `RecursiveCharacterTextSplitter`'s `chunk_size` is in **characters**, not tokens — so `500` here is *not* the same window as your from-scratch token-based chunker. This is the kind of hidden default that silently changes your retrieval quality.
- **LlamaIndex:** also ~10 lines. Its `SentenceSplitter` measures `chunk_size` in **tokens** and splits on sentence boundaries by default — closer to what Step 1 asked you to do by hand. We set `Settings.llm = None` so it does pure retrieval (no accidental OpenAI calls); tomorrow's Day 4 is where you'd plug an LLM back in for full RAG.

**The interview takeaway:** lead with the from-scratch mechanics ("I chunk with overlap, normalize, use cosine via inner product..."), *then* say "in production I'd reach for LlamaIndex or LangChain to avoid reinventing it." Knowing the primitives is what lets you reason about *why* the two frameworks return different chunks for the same query — go look: do they?
